In [1]:
import numpy as np
import pandas as pd
import altair as alt

# LOAD METADATA

In [2]:
meta=pd.read_excel("/Users/jacobg/Dropbox/shared_leif/Jacob/metadata/all_meta_JG202402.xlsx")
meta_wes = meta.dropna(subset=['wes tumor pre', 'wes tumor sg', 'wes extra FFT'], how='all')
meta_paired = meta_wes.dropna(subset='wes normal').set_index('new name', drop=False)
#bad_samples = ['DFC19_post','MGNS4_post','BIDMC3_post', 'DFC1_post']
bad_samples = ['BIDMC2_pre', 'BIDMC3_post', 'BIDMC4_pre', 'BIDMC4_post', 'BIDMC5_post', 'BIDMC6_pre', 
               'BIDMC6_post', 'BIDMC7_post', 'DFC7_pre_FFT', 'DFC9_pre', 'DFC12_post', 'DFC19_pre', 
               'DFC19_pre', 'DFC20_post', 'DFC23_post', 'MGH3_post', 'MGH5_post', 'MGNS4_pre', 
               'NWH2_pre_FFT', 'MGNS5_post', 'DFC1_pre', 'DFC19_post', 'MGNS4_post', 'BIDMC3_post', 
               'DFC1_post']
for c in ['wes tumor pre', 'wes tumor sg', 'wes extra FFT', 'wes normal']:
    meta_paired[c] = meta_paired[c].replace(bad_samples, np.nan)


tumor_meta = pd.DataFrame({'tumor': pd.concat([meta_paired['wes tumor pre'], meta_paired['wes tumor sg'], meta_paired['wes extra FFT']])}).dropna()
tumor_meta = tumor_meta.reset_index().rename(columns={'new name': 'patient'}).set_index('tumor', drop=False)
tumor_meta['patient'] = ''
tumor_meta['timing'] = ''
tumor_meta['sg_response'] = ''
tumor_meta['final_response'] = ''

for patient in meta_paired.index:
    for signature in ['wes tumor pre', 'wes tumor sg', 'wes extra FFT']:
        tumor = meta_paired.at[patient,signature]
        if pd.isna(tumor):
            continue
        tumor_meta.at[tumor,'patient'] = patient
        tumor_meta.at[tumor,'sg_response'] = meta_paired.at[patient,'post sg response']
        tumor_meta.at[tumor,'final_response'] = meta_paired.at[patient,'final response(surgery)']
        
tumor_meta['final_response_binary'] = tumor_meta['final_response'].apply(lambda x: 'pCR' if x == 'pCR' else 'RD')
tumor_meta['timing'] = tumor_meta['tumor'].apply(lambda x: 'post' if x.endswith("post") else 'pre')
tumor_meta['tissue_type'] = tumor_meta['tumor'].apply(lambda x: 'FFT' if x.endswith("FFT") else 'frozen')
tumor_meta.sort_values(by='patient', axis=0, inplace=True)

display(tumor_meta)
print(tumor_meta.shape)

,patient,tumor,timing,sg_response,final_response,final_response_binary,tissue_type
tumor,,,,,,,
BIDMC1_pre,BIDMC1,BIDMC1_pre,pre,RD,RCB-I,RD,frozen
BIDMC3_pre,BIDMC3,BIDMC3_pre,pre,pCR,pCR,pCR,frozen
BIDMC5_pre,BIDMC5,BIDMC5_pre,pre,RD,RCB-II,RD,frozen
BIDMC7_pre,BIDMC7,BIDMC7_pre,pre,RD,RCB-I,RD,frozen
BIDMC8_pre,BIDMC8,BIDMC8_pre,pre,pCR,pCR,pCR,frozen
DFC1_pre_FFT,DFC1,DFC1_pre_FFT,pre,RD,RCB-I,RD,FFT
DFC10_pre,DFC10,DFC10_pre,pre,RD,pCR,pCR,frozen
DFC11_pre,DFC11,DFC11_pre,pre,pCR,pCR,pCR,frozen
DFC13_post,DFC13,DFC13_post,post,RD,RCB-II,RD,frozen


(45, 7)


# LOAD PEAKS

In [3]:
path = '/Users/jacobg/Dropbox/shared_leif/Jacob/WES_2024/gistic/data/usable_samples/all_lesions.conf_75.txt'
peaks = pd.read_csv(path, sep='\t')
peaks['type'] = peaks['Unique Name'].apply(lambda x: x.split(' ')[0])
print(peaks.columns)
peaks

Index(['Unique Name', 'Descriptor', 'Wide Peak Limits', 'Peak Limits',
       'Region Limits', 'q values',
       'Residual q values after removing segments shared with higher peaks',
       'Broad or Focal', 'Amplitude Threshold', 'BIDMC1_pre', 'BIDMC3_pre',
       'BIDMC5_pre', 'BIDMC7_pre', 'BIDMC8_pre', 'DFC10_pre', 'DFC11_pre',
       'DFC13_post', 'DFC13_pre', 'DFC14_pre', 'DFC15_pre_FFT', 'DFC16_post',
       'DFC16_pre', 'DFC17_pre', 'DFC1_pre_FFT', 'DFC20_pre', 'DFC23_pre',
       'DFC2_post', 'DFC2_pre', 'DFC4_post', 'DFC4_pre_FFT', 'DFC6_pre',
       'DFC7_pre', 'DFC8_pre_FFT', 'MGH10_pre', 'MGH11_post', 'MGH11_pre',
       'MGH12_post', 'MGH12_pre', 'MGH13_pre', 'MGH2_pre', 'MGH3_pre',
       'MGH4_pre', 'MGH5_pre', 'MGH6_pre', 'MGH7_pre', 'MGH8_pre', 'MGH9_pre',
       'MGNS2_pre', 'MGNS5_pre', 'MGNS6_post', 'MGNS6_pre', 'NWH1_pre',
       'NWH3_post', 'NWH3_pre', 'Unnamed: 54', 'type'],
      dtype='object')


,Unique Name,Descriptor,Wide Peak Limits,Peak Limits,Region Limits,q values,Residual q values after removing segments shared with higher peaks,Broad or Focal,Amplitude Threshold,BIDMC1_pre,...,MGH9_pre,MGNS2_pre,MGNS5_pre,MGNS6_post,MGNS6_pre,NWH1_pre,NWH3_post,NWH3_pre,Unnamed: 54,type
0,Amplification Peak 1,1q21.3,chr1:150559525-150648846(probes 14837:14847),chr1:150559912-150638856(probes 14838:14846),chr1:122026459-164331374(probes 12260:16252),4.753700e-08,4.753700e-08,NaN,0: t<0.1; 1: 0.1<t< 0.9; 2: t>0.9,1.0000,...,1.0000,2.0000,1.00000,0.000000,0.000000,2.00000,2.0,2.0,NaN,Amplification
1,Amplification Peak 2,2p21,chr2:42988628-43238006(probes 28945:28969) ...,chr2:42998506-43228593(probes 28946:28968) ...,chr2:42998506-43256833(probes 28946:28971) ...,2.288600e-03,2.288600e-03,NaN,0: t<0.1; 1: 0.1<t< 0.9; 2: t>0.9,2.0000,...,2.0000,2.0000,1.00000,1.000000,0.000000,2.00000,0.0,0.0,NaN,Amplification
2,Amplification Peak 3,3p26.1,chr3:4898842-5052937(probes 48925:48941) ...,chr3:4905824-5052936(probes 48926:48940) ...,chr3:4905824-5071645(probes 48926:48943) ...,1.258100e-05,1.258100e-05,NaN,0: t<0.1; 1: 0.1<t< 0.9; 2: t>0.9,0.0000,...,0.0000,0.0000,0.00000,1.000000,2.000000,0.00000,0.0,0.0,NaN,Amplification
3,Amplification Peak 4,3q25.31,chr3:157086913-157210864(probes 63970:63984),chr3:157096894-157210462(probes 63971:63983),chr3:157086913-157289724(probes 63970:63995),2.604000e-10,2.604000e-10,NaN,0: t<0.1; 1: 0.1<t< 0.9; 2: t>0.9,2.0000,...,1.0000,0.0000,2.00000,2.000000,2.000000,2.00000,0.0,0.0,NaN,Amplification
4,Amplification Peak 5,5q35.3,chr5:179615482-179633437(probes 104124:104130),chr5:179615483-179633420(probes 104125:104129),chr5:179612138-179643470(probes 104122:104132),2.864600e-03,2.864600e-03,NaN,0: t<0.1; 1: 0.1<t< 0.9; 2: t>0.9,0.0000,...,0.0000,0.0000,0.00000,0.000000,1.000000,0.00000,0.0,2.0,NaN,Amplification
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,Deletion Peak 12 - CN values,14q32.33,chr14:105853499-107043718(probes 222548:222576),chr14:105905871-107043718(probes 222554:222576),chr14:103274069-107043718(probes 222272:222576),1.457600e-06,1.439300e-06,NaN,Actual Copy Change Given,-1.2929,...,-0.5000,3.6569,-1.29290,-0.007113,-0.666670,-1.00000,0.0,0.0,NaN,Deletion
58,Deletion Peak 13 - CN values,15q15.1,chr15:41749080-41983658(probes 224791:224813),chr15:41759016-41983658(probes 224792:224813),chr15:36768006-44225290(probes 224289:225038),5.905300e-04,5.860400e-04,NaN,Actual Copy Change Given,-0.5000,...,-1.0000,0.0000,-1.29290,-0.007113,0.002122,-1.00000,0.0,0.0,NaN,Deletion
59,Deletion Peak 14 - CN values,16p13.3,chr16:1-1368804(probes 230810:230932) ...,chr16:839687-1232938(probes 230876:230916) ...,chr16:1-3024776(probes 230810:231113) ...,2.932900e-06,2.932900e-06,NaN,Actual Copy Change Given,-0.5000,...,-1.2929,0.0000,0.00000,-1.292900,-1.292900,-0.10569,1.0,0.0,NaN,Deletion
60,Deletion Peak 15 - CN values,17p11.2,chr17:21249279-27073782(probes 241835:242038),chr17:21269108-22544596(probes 241837:241964),chr17:21298852-21318783(probes 241840:241841),1.808700e-03,1.808400e-03,NaN,Actual Copy Change Given,-1.0000,...,-1.0000,-1.0000,-0.66667,-0.597020,0.754190,0.00000,0.0,0.0,NaN,Deletion


# GROUPS

In [4]:
sample_columns = [x for x in peaks.columns if '_' in x]

# get matched samples
patients_with_two_samples = tumor_meta['patient'].value_counts()
patients_with_two_samples = patients_with_two_samples[patients_with_two_samples == 2].index.tolist()
paired_meta = tumor_meta[tumor_meta['patient'].isin(patients_with_two_samples)]
paired_columns = [x for x in sample_columns if x in paired_meta['tumor'].values]

pre_pcr_drop = [tumor for tumor in sample_columns if (tumor_meta.at[tumor, 'timing'] != 'pre') or (tumor_meta.at[tumor, 'sg_response'] != 'pCR')]
pre_RD_drop = [tumor for tumor in sample_columns if (tumor_meta.at[tumor, 'timing'] != 'pre') or (tumor_meta.at[tumor, 'sg_response'] != 'RD')]
post_RD_drop = [tumor for tumor in sample_columns if (tumor_meta.at[tumor, 'timing'] != 'post') or (tumor_meta.at[tumor, 'sg_response'] != 'RD')]
pre_drop = [tumor for tumor in sample_columns if (tumor_meta.at[tumor, 'timing'] != 'pre')]
post_drop = [tumor for tumor in sample_columns if (tumor_meta.at[tumor, 'timing'] != 'post')]
pre_paired_drop = [tumor for tumor in sample_columns if ((tumor_meta.at[tumor, 'timing'] != 'pre') or (tumor not in paired_columns))]
post_paired_drop = [tumor for tumor in sample_columns if (tumor_meta.at[tumor, 'timing'] != 'post') or (tumor not in paired_columns)]


pre_pcr_peaks = peaks[peaks['Amplitude Threshold'] != 'Actual Copy Change Given'].drop(columns = pre_pcr_drop).copy()
pre_RD_peaks = peaks[peaks['Amplitude Threshold'] != 'Actual Copy Change Given'].drop(columns = pre_RD_drop).copy()
post_RD_peaks = peaks[peaks['Amplitude Threshold'] != 'Actual Copy Change Given'].drop(columns = post_RD_drop).copy()
pre_peaks = peaks[peaks['Amplitude Threshold'] != 'Actual Copy Change Given'].drop(columns = pre_drop).copy()
post_peaks = peaks[peaks['Amplitude Threshold'] != 'Actual Copy Change Given'].drop(columns = post_drop).copy()
pre_paired_peaks = peaks[peaks['Amplitude Threshold'] != 'Actual Copy Change Given'].drop(columns = pre_paired_drop).copy()
post_paired_peaks = peaks[peaks['Amplitude Threshold'] != 'Actual Copy Change Given'].drop(columns = post_paired_drop).copy()

pre_pcr_peaks.name = 'Pretreatment pCR'
pre_RD_peaks.name = 'Pretreatment RD'
post_RD_peaks.name = 'Posttreatment RD'
pre_peaks.name = 'Pretreatment'
post_peaks.name = 'posttreatment'
pre_paired_peaks.name = 'Paired Pretreatment'
post_paired_peaks.name = 'Paired Posttreatment'

sample_sizes = {}
for d in [pre_pcr_peaks, pre_RD_peaks,pre_RD_peaks, post_RD_peaks, pre_peaks, post_peaks, pre_paired_peaks, post_paired_peaks]:
    sample_sizes[d.name] = len([x for x in d.columns if x in sample_columns])
    
print(sample_sizes)


{'Pretreatment pCR': 12, 'Pretreatment RD': 25, 'Posttreatment RD': 8, 'Pretreatment': 37, 'posttreatment': 8, 'Paired Pretreatment': 8, 'Paired Posttreatment': 8}


# BAR PLOTS

## S1C

In [5]:
for dataframes in [[pre_pcr_peaks, pre_RD_peaks], [pre_RD_peaks, post_RD_peaks], [pre_peaks, post_peaks], [pre_paired_peaks, post_paired_peaks]]:
    for idx, d in enumerate(dataframes):
        # Process the dataframe
        cols = [x for x in d.columns if x in sample_columns]
        d['minor'] = (d[cols] == 1).sum(axis=1)
        d['major'] = (d[cols] == 2).sum(axis=1)
        d['count'] = d['minor'] + d['major']
        d['major_freq'] = d['major'] / len(cols)
        d['minor_freq'] = d['minor'] / len(cols)
        d['frequency'] = d['count'] / len(cols)


group_fracs = pd.DataFrame( data={'Descriptor':pre_peaks['Descriptor'],  'Unique Name': pre_peaks['Unique Name'].values})
group_fracs['Peak_Type'] = group_fracs['Unique Name'].apply(lambda x: x.split(' ')[0])
group_fracs = group_fracs.merge(pre_pcr_peaks[['Unique Name', 'frequency']], how='outer', on='Unique Name').rename(columns={'frequency': 'pre-pCR'})
group_fracs = group_fracs.merge(pre_RD_peaks[['Unique Name', 'frequency']], how='outer', on='Unique Name').rename(columns={'frequency': 'pre-RD'})
display(group_fracs)

# Reshape data for diverging plot
plot_data = []

for _, row in group_fracs.iterrows():
    # Add pre-pCR as negative values
    plot_data.append({
        'Peak': row['Descriptor'],
        'Value': -row['pre-pCR'],  # Negative for left side
        'Group': 'pre-pCR',
        'DisplayValue': row['pre-pCR'],
        'DisplayText': f"{row['pre-pCR']:.2f}",
        'Peak_Type': row['Peak_Type']
    })
    
    # Add pre-RD as positive values 
    plot_data.append({
        'Peak': row['Descriptor'],
        'Value': row['pre-RD'],  # Positive for right side
        'Group': 'pre-RD', 
        'DisplayValue': row['pre-RD'],
        'DisplayText': f"{row['pre-RD']:.2f}",
        'Peak_Type': row['Peak_Type']
    })

plot_df = pd.DataFrame(plot_data)

# Calculate total for sorting
peak_totals = plot_df.groupby('Peak')['DisplayValue'].sum().reset_index()
peak_order = peak_totals.sort_values('DisplayValue', ascending=False)['Peak'].tolist()

# Create the bar chart
bars = alt.Chart(plot_df).mark_bar().encode(
    y=alt.Y('Peak:N', 
            title='Peaks',
            sort=peak_order,
            axis=alt.Axis(labelLimit=0)),
    x=alt.X('Value:Q', 
            title='Proportion',
            scale=alt.Scale(domain=[-1.2, 1.2]),
            axis=alt.Axis(
                values=[-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1],
                labelExpr="abs(datum.value) == 0 ? '0' : abs(datum.value)"
            )),
    color=alt.Color('Group:N', 
                   scale=alt.Scale(
                       domain=['pre-pCR', 'pre-RD'], 
                       range=["#CC0C00FF", "#5C88DAFF"]
                   ))
)

# Add text labels
text = alt.Chart(plot_df).mark_text(
    align=alt.expr('datum.Value < 0 ? "right" : "left"'),
    baseline='middle',
    dx=alt.expr('datum.Value < 0 ? -5 : 5'),
    fontSize=10,
    fontWeight='bold'
).encode(
    y=alt.Y('Peak:N', sort=peak_order),
    x=alt.X('Value:Q'),
    text='DisplayText:N',
    color=alt.value('black')
)

# Add center line
rule = alt.Chart(plot_df).mark_rule(color='black', strokeWidth=2).encode(
    x=alt.value(0)
)

# Layer charts, set properties, then apply faceting
chart = (bars + text + rule).properties(
    width=300,  # Width per panel
    height=300
).facet(
    column=alt.Column('Peak_Type:N',
                     header=alt.Header(title="Peak Type", 
                                     titleFontSize=14,
                                     labelFontSize=12))
).resolve_scale(
    y='independent'  # Allow independent y-scales for each panel
).properties(
    title='Deletion vs Amplification Peaks: pre-pCR vs pre-RD Comparison'
).configure_axis(
    grid=True,
    gridDash=[3, 3]
).configure_header(
    titleColor='black',
    titleFontSize=14,
    labelFontSize=12
).configure_view(
    stroke='transparent'
).configure_legend(
    title=None,
    orient='bottom',
    labelFontSize=12
)

# Display the chart
#chart.save('/Users/jacobg/Dropbox/shared_leif/neoSTAR A1 manuscript prep/exome_Jacob/plots/gistic_peaks.pdf')
chart

,Descriptor,Unique Name,Peak_Type,pre-pCR,pre-RD
0,1q21.3,Amplification Peak 1,Amplification,0.833333,0.84
1,2p21,Amplification Peak 2,Amplification,0.750000,0.68
2,3p26.1,Amplification Peak 3,Amplification,0.583333,0.40
3,3q25.31,Amplification Peak 4,Amplification,0.750000,0.60
4,5q35.3,Amplification Peak 5,Amplification,0.416667,0.36
5,6p22.2,Amplification Peak 6,Amplification,0.666667,0.68
6,8q24.11,Amplification Peak 7,Amplification,0.916667,0.68
7,10p15.2,Amplification Peak 8,Amplification,0.666667,0.72
8,11p13,Amplification Peak 9,Amplification,0.416667,0.40
9,12p13.1,Amplification Peak 10,Amplification,0.833333,0.60


alt.FacetChart(...)

## S11B

In [6]:
group_fracs = pd.DataFrame( data={'Descriptor':pre_peaks['Descriptor'],  'Unique Name': pre_peaks['Unique Name'].values})
group_fracs['Peak_Type'] = group_fracs['Unique Name'].apply(lambda x: x.split(' ')[0])
group_fracs = group_fracs.merge(pre_paired_peaks[['Unique Name', 'frequency']], how='outer', on='Unique Name').rename(columns={'frequency': 'pre-RD'})
group_fracs = group_fracs.merge(post_paired_peaks[['Unique Name', 'frequency']], how='outer', on='Unique Name').rename(columns={'frequency': 'post-RD'})

display(group_fracs)

# Reshape data for diverging plot
plot_data = []

for _, row in group_fracs.iterrows():
    # Add pre-RD as negative values
    plot_data.append({
        'Peak': row['Descriptor'],
        'Value': -row['pre-RD'],  # Negative for left side
        'Group': 'pre-RD',
        'DisplayValue': row['pre-RD'],
        'DisplayText': f"{row['pre-RD']:.2f}",
        'Peak_Type': row['Peak_Type']
    })
    
    # Add post-RD as positive values
    plot_data.append({
        'Peak': row['Descriptor'],
        'Value': row['post-RD'],  # Positive for right side
        'Group': 'post-RD', 
        'DisplayValue': row['post-RD'],
        'DisplayText': f"{row['post-RD']:.2f}",
        'Peak_Type': row['Peak_Type']
    })

plot_df = pd.DataFrame(plot_data)

# Calculate total for sorting
peak_totals = plot_df.groupby('Peak')['DisplayValue'].sum().reset_index()
peak_order = peak_totals.sort_values('DisplayValue', ascending=False)['Peak'].tolist()

# Create the bar chart
bars = alt.Chart(plot_df).mark_bar().encode(
    y=alt.Y('Peak:N', 
            title='Peaks',
            sort=peak_order,
            axis=alt.Axis(labelLimit=0)),
    x=alt.X('Value:Q', 
            title='Proportion',
            scale=alt.Scale(domain=[-1.2, 1.2]),
            axis=alt.Axis(
                values=[-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1],
                labelExpr="abs(datum.value) == 0 ? '0' : abs(datum.value)"
            )),
    color=alt.Color('Group:N', 
                   scale=alt.Scale(
                       domain=['pre-RD', 'post-RD'], 
                       range=["#5C88DAFF", "#FFCD00FF"]
                   ))
)

# Add text labels
text = alt.Chart(plot_df).mark_text(
    align=alt.expr('datum.Value < 0 ? "right" : "left"'),
    baseline='middle',
    dx=alt.expr('datum.Value < 0 ? -5 : 5'),
    fontSize=10,
    fontWeight='bold'
).encode(
    y=alt.Y('Peak:N', sort=peak_order),
    x=alt.X('Value:Q'),
    text='DisplayText:N',
    color=alt.value('black')
)

# Add center line
rule = alt.Chart(plot_df).mark_rule(color='black', strokeWidth=2).encode(
    x=alt.value(0)
)

# Layer charts, set properties, then apply faceting
chart = (bars + text + rule).properties(
    width=300,  # Width per panel
    height=300
).facet(
    column=alt.Column('Peak_Type:N',
                     header=alt.Header(title="Peak Type", 
                                     titleFontSize=14,
                                     labelFontSize=12))
).resolve_scale(
    y='independent'  # Allow independent y-scales for each panel
).properties(
    title='Deletion vs Amplification Peaks: pre-RD vs post-RD Comparison'
).configure_axis(
    grid=True,
    gridDash=[3, 3]
).configure_header(
    titleColor='black',
    titleFontSize=14,
    labelFontSize=12
).configure_view(
    stroke='transparent'
).configure_legend(
    title=None,
    orient='bottom',
    labelFontSize=12
)

# Display the chart
#chart.save('/Users/jacobg/Dropbox/shared_leif/neoSTAR A1 manuscript prep/exome_Jacob/plots/gistic_peaks_paired.pdf')
chart

,Descriptor,Unique Name,Peak_Type,pre-RD,post-RD
0,1q21.3,Amplification Peak 1,Amplification,0.625,0.625
1,2p21,Amplification Peak 2,Amplification,0.500,0.625
2,3p26.1,Amplification Peak 3,Amplification,0.625,0.500
3,3q25.31,Amplification Peak 4,Amplification,0.625,0.625
4,5q35.3,Amplification Peak 5,Amplification,0.500,0.125
5,6p22.2,Amplification Peak 6,Amplification,0.500,0.625
6,8q24.11,Amplification Peak 7,Amplification,0.500,0.875
7,10p15.2,Amplification Peak 8,Amplification,0.375,0.375
8,11p13,Amplification Peak 9,Amplification,0.250,0.500
9,12p13.1,Amplification Peak 10,Amplification,0.750,0.750


alt.FacetChart(...)